<a href="https://colab.research.google.com/github/somaiah-ui/AI-Summarizer/blob/main/AI_Summarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q -U \
    langchain \
    langchain-core \
    langchain-huggingface \
    transformers \
    accelerate \
    gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.7/163.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.0/572.0 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 4.9 MB/s eta 0:00:00


In [2]:
import torch
import gradio as gr

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFacePipeline


# ---------------------------------------------
# 1. Check GPU
# ---------------------------------------------

device = 0 if torch.cuda.is_available() else -1

print(
    "Using GPU" if torch.cuda.is_available()
    else "Using CPU"
)


# ---------------------------------------------
# 2. Load AI model using LangChain
# ---------------------------------------------

llm = HuggingFacePipeline.from_model_id(

    model_id="Qwen/Qwen2.5-0.5B-Instruct",

    task="text-generation",

    device=device,

    pipeline_kwargs={
        "max_new_tokens": 150,
        "do_sample": False,
        "return_full_text": False
    }

)

print("Model loaded successfully!")


# ---------------------------------------------
# 3. Create LangChain Prompt
# ---------------------------------------------

prompt = PromptTemplate.from_template(
"""
You are an AI text summarizer.

Summarize the following text in simple,
clear language.

Keep only the important information.

TEXT:

{text}

SUMMARY:
"""
)


# ---------------------------------------------
# 4. Create LangChain Chain
# ---------------------------------------------

chain = (
    prompt
    | llm
    | StrOutputParser()
)


# ---------------------------------------------
# 5. Summarizer Function
# ---------------------------------------------

def summarize_text(text):

    if not text.strip():

        return "Please enter some text."

    try:

        result = chain.invoke({
            "text": text
        })

        return result.strip()

    except Exception as e:

        return f"Error: {e}"


# ---------------------------------------------
# 6. Gradio Website
# ---------------------------------------------

with gr.Blocks() as app:

    gr.Markdown(
"""
# 📝 LangChain AI Summarizer

Paste a paragraph, article, or notes below.

The AI will summarize the text using:

**LangChain + Hugging Face + Qwen**
"""
    )


    input_text = gr.Textbox(

        label="Enter Your Text",

        placeholder="Paste your text here...",

        lines=12

    )


    summarize_button = gr.Button(
        "Summarize"
    )


    output_text = gr.Textbox(

        label="AI Summary",

        lines=6

    )


    summarize_button.click(

        fn=summarize_text,

        inputs=input_text,

        outputs=output_text

    )


# ---------------------------------------------
# 7. Start Application
# ---------------------------------------------

app.launch(
    share=True
)

Using GPU


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded successfully!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://307f8876516a9ecbee.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
